In [ ]:
# Task B first baselines: does the evidence SUPPORT, CONTRADICT, or not settle (UNSUPPORTED) each claim?
# Simple baselines come first; an NLI model is the next step. Similar wording is not proof of support.
import sys, json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sentence_transformers import SentenceTransformer
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()  # hide model-loading reports

# Find the repository root (works from any folder) and import the shared project code.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/task_a").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from task_a.leakage import comparison_key
from task_b.data import LABELS, load_claims

In [ ]:
# Load the settings and the prepared train/validation/test files (format: src/task_b/data.py, docs/task_b_plan.md).
config = json.loads((ROOT / "configs/task_b_fever_baselines.json").read_text(encoding="utf-8"))
SEED = config["seed"]
np.random.seed(SEED)
data_dir = ROOT / config["data_dir"]
if not data_dir.is_dir():
    raise FileNotFoundError(f"{data_dir} not found: prepare the three JSONL files first (see docs/task_b_plan.md).")
splits = {split: pd.DataFrame(load_claims(data_dir / file_name)) for split, file_name in config["files"].items()}

In [ ]:
# Sanity checks: every label occurs in train, and no claim appears in two splits (case and spacing ignored).
assert set(splits["train"]["label"]) == set(LABELS)
claim_keys = {split: set(frame["claim"].map(comparison_key)) for split, frame in splits.items()}
assert not claim_keys["train"] & claim_keys["validation"], "a claim appears in train and validation"
assert not (claim_keys["train"] | claim_keys["validation"]) & claim_keys["test"], "a test claim appears in train/validation"
print("All checks passed:", {split: len(frame) for split, frame in splits.items()})

In [ ]:
# Dataset summary: claims per label and split. Evidence length per label is printed as a shortcut check:
# if one label had much shorter evidence, a model could guess it without verifying anything.
summary = pd.DataFrame({split: frame["label"].value_counts() for split, frame in splits.items()}).reindex(LABELS)
summary.loc["Total"] = summary.sum()
evidence_words = pd.concat(splits.values()).assign(words=lambda f: f["evidence"].str.split().str.len()).groupby("label")["words"].median()
print("Median evidence length in words:", {label: int(evidence_words[label]) for label in LABELS})
summary

In [ ]:
# Baseline 0: always predict the most common training label. Any useful verifier must beat this.
majority_label = splits["train"]["label"].mode()[0]
predictions = {"Majority class": {split: np.array([majority_label] * len(splits[split])) for split in ["validation", "test"]}}
print("Most common training label:", majority_label)

In [ ]:
# Baseline 1: claim only, evidence ignored. A high score means labels can be guessed from claim wording alone,
# which a real verifier must clearly beat. C is chosen on validation macro-F1 only.
vectorizer = TfidfVectorizer(ngram_range=tuple(config["tfidf"]["ngram_range"]), min_df=config["tfidf"]["min_df"], sublinear_tf=True)
X_train = vectorizer.fit_transform(splits["train"]["claim"])
X_validation = vectorizer.transform(splits["validation"]["claim"])
X_test = vectorizer.transform(splits["test"]["claim"])
best_c_claim, best_f1 = None, -1.0
for c in config["c_grid"]:
    classifier = LogisticRegression(C=c, max_iter=2000, random_state=SEED).fit(X_train, splits["train"]["label"])
    val_f1 = f1_score(splits["validation"]["label"], classifier.predict(X_validation), average="macro")
    print(f"C = {c:<5}  validation macro-F1 = {val_f1:.3f}")
    if val_f1 > best_f1:
        best_c_claim, best_f1, claim_only = c, val_f1, classifier
predictions["Claim only (TF-IDF)"] = {"validation": claim_only.predict(X_validation), "test": claim_only.predict(X_test)}
print(f"Selected C = {best_c_claim}")

In [ ]:
# Sentence embeddings for claims and evidence; their cosine similarity measures topic overlap, not agreement.
encoder = SentenceTransformer(config["embedding_model"])
similarity = {}
for split, frame in splits.items():
    claim_vectors = encoder.encode(frame["claim"].tolist(), batch_size=64, normalize_embeddings=True)
    evidence_vectors = encoder.encode(frame["evidence"].tolist(), batch_size=64, normalize_embeddings=True)
    similarity[split] = (claim_vectors * evidence_vectors).sum(axis=1).reshape(-1, 1)  # cosine: vectors have length 1
print("Median validation similarity by label:",
      {label: round(float(np.median(similarity["validation"][splits["validation"]["label"] == label])), 3) for label in LABELS})

In [ ]:
# Baseline 2 (roadmap embedding baseline): a logistic regression maps the single similarity score to a label.
# Fit on train; C chosen on validation macro-F1 only.
best_c_similarity, best_f1 = None, -1.0
for c in config["c_grid"]:
    classifier = LogisticRegression(C=c, max_iter=2000, random_state=SEED).fit(similarity["train"], splits["train"]["label"])
    val_f1 = f1_score(splits["validation"]["label"], classifier.predict(similarity["validation"]), average="macro")
    print(f"C = {c:<5}  validation macro-F1 = {val_f1:.3f}")
    if val_f1 > best_f1:
        best_c_similarity, best_f1, similarity_model = c, val_f1, classifier
predictions["Embedding similarity"] = {split: similarity_model.predict(similarity[split]) for split in ["validation", "test"]}
print(f"Selected C = {best_c_similarity}")

In [ ]:
# Results: per-class precision (P), recall (R) and F1 plus macro-F1 and accuracy. A class that is never
# predicted gets precision 0 (scikit-learn convention), e.g. for the majority baseline.
rows = {}
for model, split_predictions in predictions.items():
    for split in ["validation", "test"]:
        gold, predicted = splits[split]["label"], split_predictions[split]
        precision, recall, f1, _ = precision_recall_fscore_support(gold, predicted, labels=list(LABELS), zero_division=0)
        row = {"Accuracy": accuracy_score(gold, predicted), "Macro-F1": f1.mean()}
        for label, p, r, f in zip(LABELS, precision, recall, f1):
            row[f"{label} P"], row[f"{label} R"], row[f"{label} F1"] = p, r, f
        rows[(split, model)] = row
results = pd.DataFrame(rows).T
results.style.format("{:.3f}")

In [ ]:
# Test-set confusion matrices: rows are the true labels, columns are the predictions.
fig, axes = plt.subplots(1, len(predictions), figsize=(5.5 * len(predictions), 5))
for ax, (model, split_predictions) in zip(axes, predictions.items()):
    counts = confusion_matrix(splits["test"]["label"], split_predictions["test"], labels=list(LABELS))
    ConfusionMatrixDisplay(counts, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=",d", colorbar=False)
    ax.set_title(f"{model} (test, n = {counts.sum():,})")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.tick_params(axis="x", labelrotation=20)
fig.tight_layout()

In [ ]:
# Save settings, metrics, predictions and the figure to a new timestamped folder.
run_dir = ROOT / config["output_dir"] / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
run_dir.mkdir(parents=True, exist_ok=False)
record = {"created_utc": datetime.now(timezone.utc).isoformat(), "interface": "notebooks/task_b_fever_baselines.ipynb",
          "config": config, "seed": SEED, "selected_C": {"claim_only": best_c_claim, "embedding_similarity": best_c_similarity},
          "versions": {"python": sys.version.split()[0], "scikit_learn": sklearn.__version__}}
(run_dir / "run.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
results.to_csv(run_dir / "metrics.csv")
saved = [pd.DataFrame({"id": splits[split]["id"], "split": split, "gold": splits[split]["label"],
                       **{model: split_predictions[split] for model, split_predictions in predictions.items()}})
         for split in ["validation", "test"]]
pd.concat(saved).to_csv(run_dir / "predictions.csv", index=False)
fig.savefig(run_dir / "confusion_matrices.png", dpi=150, bbox_inches="tight")
print("Results saved to", run_dir.relative_to(ROOT))

In [ ]:
# Conclusion generated from the results table above.
test_scores = results.loc["test"]
print("Test macro-F1:", ", ".join(f"{model} {score:.3f}" for model, score in test_scores["Macro-F1"].items()))
print(f"Claim-only minus majority: {test_scores.loc['Claim only (TF-IDF)', 'Macro-F1'] - test_scores.loc['Majority class', 'Macro-F1']:+.3f} "
      "macro-F1 (how much the claim wording alone reveals about the label).")
print(f"Embedding similarity, test F1 per label: " + ", ".join(
    f"{label} {test_scores.loc['Embedding similarity', f'{label} F1']:.3f}" for label in LABELS))
print("Next step: an NLI model (entailment / contradiction / neutral), evaluated on the same splits (docs/task_b_plan.md).")